# Training Face Recognition dengan MediaPipe Landmark + Fixed Pipeline

Notebook ini adalah versi perbaikan dari `train_mediapipe_conv1d.ipynb`.
Perubahan utama:
- Landmark di-align (translate nose tip, scale jarak antar mata, rotate mata horizontal).
- Normalisasi Z-Score di-fit dari training set saja (tidak ada data leakage).
- Augmentasi gambar sebelum ekstraksi landmark.
- Model PointNet-style (1x1 Conv1D + GlobalMaxPooling) untuk mengurangi overfit.
- Deteksi wajah asing menggunakan **embedding + centroid distance**, bukan hanya softmax threshold.
- Preprocessing CLAHE disamakan antara training dan inference.


In [ ]:

import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import pickle
import random
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print(tf.__version__)


## 1. Preprocessing Gambar (CLAHE)

In [ ]:

# ==========================================
# PREPROCESSING (CLAHE)
# ==========================================

def apply_clahe(img):
    """Meningkatkan kontras gambar menggunakan CLAHE agar
    MediaPipe lebih akurat mendeteksi titik wajah,
    terutama pada kondisi pencahayaan buruk."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.merge((l, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

print("[INFO] Fungsi CLAHE siap.")


## 2. Ekstraksi Landmark + Align + Augmentasi

`align_landmarks()` membuat landmark menjadi **translation, scale, dan rotation invariant**:
- Nose tip dipindahkan ke origin.
- Seluruh landmark di-scale oleh jarak antar mata (outer eye corners).
- Wajah di-rotate agar garis antar mata horizontal.

Selain gambar asli, kita generate 2 varian augmented (brightness/contrast + rotasi/scale) untuk menambah variasi pose & cahaya.


In [ ]:

# ==========================================
# EKSTRAKSI, ALIGN, DAN AUGMENTASI
# ==========================================

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5
)

# Indeks landmark referensi MediaPipe (468 landmark)
NOSE_TIP = 1
LEFT_EYE = 33
RIGHT_EYE = 263


def align_landmarks(landmarks):
    """Normalisasi geometris landmark wajah.
    Input/output shape: (468, 3)
    """
    lm = np.array(landmarks, dtype=np.float32)

    # 1. Translate: nose tip ke origin
    lm -= lm[NOSE_TIP]

    # 2. Scale: jarak antar mata = 1
    eye_dist = np.linalg.norm(lm[LEFT_EYE] - lm[RIGHT_EYE])
    if eye_dist > 1e-6:
        lm /= eye_dist

    # 3. Rotate: mata jadi horizontal di bidang XY
    left = lm[LEFT_EYE].copy()
    right = lm[RIGHT_EYE].copy()
    angle = np.arctan2(right[1] - left[1], right[0] - left[0])
    c, s = np.cos(-angle), np.sin(-angle)
    R = np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ], dtype=np.float32)
    lm = lm @ R.T

    return lm


def augment_image(img):
    """Augmentasi sederhana: brightness/contrast + rotasi + scale."""
    h, w = img.shape[:2]

    # brightness / contrast
    alpha = random.uniform(0.7, 1.3)
    beta = random.randint(-30, 30)
    aug = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    # rotasi & scale
    angle = random.uniform(-12, 12)
    scale = random.uniform(0.9, 1.1)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    aug = cv2.warpAffine(aug, M, (w, h), borderValue=(128, 128, 128))

    return aug


dataset_path = "Dataset/Dataset_wajah"
X_raw, Y = [], []
gagal_deteksi = 0

print("[INFO] Memulai ekstraksi landmark (CLAHE + align + augmentasi)...\n")

for person_name in sorted(os.listdir(dataset_path)):
    person_dir = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_dir):
        continue

    count = 0
    for image_name in sorted(os.listdir(person_dir)):
        image_path = os.path.join(person_dir, image_name)
        img = cv2.imread(image_path)
        if img is None:
            continue

        # Proses gambar asli + 2 varian augmented
        variants = [img, augment_image(img), augment_image(img)]
        for variant in variants:
            enhanced = apply_clahe(variant)
            rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb)

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]
                lms = np.array(
                    [[lm.x, lm.y, lm.z] for lm in face_landmarks.landmark],
                    dtype=np.float32
                )
                lms = align_landmarks(lms)
                X_raw.append(lms)
                Y.append(person_name)
                count += 1
            else:
                gagal_deteksi += 1

    print(f"  {person_name}: {count} sampel")

X = np.array(X_raw, dtype=np.float32)
Y = np.array(Y)

print(f"\nTotal sampel     : {len(X)}")
print(f"Gagal deteksi    : {gagal_deteksi}")
print(f"Shape X          : {X.shape}")


## 3. Visualisasi Landmark Hasil Align

In [ ]:

# ==========================================
# VISUALISASI
# ==========================================

sample_person = sorted(os.listdir(dataset_path))[0]
sample_dir = os.path.join(dataset_path, sample_person)
sample_img_name = sorted(os.listdir(sample_dir))[0]
sample_img_path = os.path.join(sample_dir, sample_img_name)

img_sample = cv2.imread(sample_img_path)
img_enhanced = apply_clahe(img_sample)
img_rgb = cv2.cvtColor(img_enhanced, cv2.COLOR_BGR2RGB)
results_viz = face_mesh.process(img_rgb)

if results_viz.multi_face_landmarks:
    face_lm = results_viz.multi_face_landmarks[0]
    h, w, _ = img_rgb.shape

    # raw pixel coordinates
    xs = [lm.x * w for lm in face_lm.landmark]
    ys = [lm.y * h for lm in face_lm.landmark]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(cv2.cvtColor(img_sample, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Gambar Asli")
    axes[0].axis("off")

    axes[1].imshow(img_rgb)
    axes[1].scatter(xs, ys, c="lime", s=1, alpha=0.7)
    axes[1].set_title("468 Titik MediaPipe")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("[WARNING] Wajah tidak terdeteksi pada sampel.")


## 4. Split Data & Normalisasi Z-Score

**Penting:** split dulu, baru hitung `mean` dan `std` dari training set saja untuk menghindari data leakage.


In [ ]:

# ==========================================
# SPLIT & NORMALISASI Z-SCORE
# ==========================================

x_train, x_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

# Hitung statistik dari TRAINING saja
mean = x_train.mean(axis=0)
std = x_train.std(axis=0) + 1e-8

x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

np.save("mean_landmark.npy", mean)
np.save("std_landmark.npy", std)

print(f"Train shape : {x_train.shape}")
print(f"Test shape  : {x_test.shape}")
print(f"Mean shape  : {mean.shape}, Std shape: {std.shape}")


## 5. Label Encoding

In [ ]:

# ==========================================
# LABEL ENCODING
# ==========================================

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
num_classes = len(le.classes_)

y_train_cat = keras.utils.to_categorical(y_train_enc, num_classes=num_classes)
y_test_cat = keras.utils.to_categorical(y_test_enc, num_classes=num_classes)

with open('label_encoder_last.pickle', 'wb') as f:
    pickle.dump(le, f)

print(f"Jumlah kelas: {num_classes}\n")
for idx, name in enumerate(le.classes_):
    print(f"  {idx}: {name}")


## 6. Arsitektur Model

Menggunakan **PointNet-style 1D CNN**:
- `Conv1D(kernel_size=1)` memproses fitur tiap landmark secara independen.
- `GlobalMaxPooling1D` mengurangi jumlah parameter drastis dan lebih tahan terhadap variasi posisi.
- Dense layer terakhir sebelum softmax digunakan sebagai **embedding** untuk deteksi unknown.


In [ ]:

# ==========================================
# ARSITEKTUR MODEL
# ==========================================

def build_model(input_shape=(468, 3), num_classes=20):
    inputs = layers.Input(shape=input_shape)

    # PointNet-style per-point feature learning
    x = layers.Conv1D(64, 1, activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 1, activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(256, 1, activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)

    x = layers.GlobalMaxPooling1D()(x)

    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)

    # Embedding layer (digunakan untuk unknown detection)
    embedding = layers.Dense(128, activation='relu', name='embedding')(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='softmax')(embedding)

    return models.Model(inputs, outputs, name='mediapipe_pointnet')


model = build_model(input_shape=(468, 3), num_classes=num_classes)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


## 7. Training dengan Early Stopping

In [ ]:

# ==========================================
# TRAINING
# ==========================================

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]
)


## 8. Simpan Model + Embedding Centroid & Threshold

Untuk deteksi wajah asing, kita:
1. Ambil embedding dari layer sebelum softmax.
2. Hitung centroid tiap kelas dari data training.
3. Simpan threshold = persentil ke-95 jarak tiap sample ke centroid-nya. Jika jarak inference melebihi threshold → `Unknown`.


In [ ]:

# ==========================================
# SAVE MODEL & COMPUTE CENTROIDS
# ==========================================

model.save("model_cnn_last.keras")

# Model untuk ekstrak embedding
embedding_model = models.Model(
    inputs=model.input,
    outputs=model.get_layer('embedding').output
)

# Hitung embedding training & test
train_emb = embedding_model.predict(x_train, verbose=0)
test_emb = embedding_model.predict(x_test, verbose=0)

# Centroid & threshold per kelas.
# Threshold dihitung dari jarak sample TEST/VALIDASI ke centroid-nya,
# lalu dikali slack 1.3 supaya wajah asli di webcam tidak terlalu sering
# tertolak sebagai Unknown.
centroids, thresholds = compute_centroids_and_thresholds(
    train_emb, y_train_enc, num_classes,
    percentile=95, slack=1.3,
    val_embeddings=test_emb, val_labels=y_test_enc
)

np.save("face_centroids.npy", centroids)
np.save("face_thresholds.npy", thresholds)

print("[INFO] Model & centroid tersimpan.")
print(f"Thresholds per kelas (validasi x 1.3):")
for i, name in enumerate(le.classes_):
    print(f"  {name:25s}: {thresholds[i]:.4f}")


## 9. Evaluasi Akurasi & Kurva

In [ ]:

# ==========================================
# EVALUASI
# ==========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Acc')
axes[0].plot(history.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Akurasi')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

loss, acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"\n[HASIL] Akurasi Test: {acc*100:.2f}%")


## 10. Confusion Matrix & Classification Report

In [ ]:

# ==========================================
# CONFUSION MATRIX & REPORT
# ==========================================

y_pred = model.predict(x_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = y_test_enc

plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_true_classes, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_true_classes, y_pred_classes, target_names=le.classes_))


## 11. Inference Webcam dengan Unknown Detection

Perubahan di inference:
- Preprocessing CLAHE sama seperti training.
- Landmark di-align sebelum masuk model.
- Prediksi final pakai **embedding distance ke centroid**, bukan cuma softmax confidence.
- Kalau jarak melebihi threshold kelas terdekat → `Unknown`.


In [ ]:

# ==========================================
# WEBCAM INFERENCE
# ==========================================

import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
import pickle
from collections import deque

model = tf.keras.models.load_model("model_cnn_last.keras")

with open("label_encoder_last.pickle", "rb") as f:
    le = pickle.load(f)

mean = np.load("mean_landmark.npy")
std = np.load("std_landmark.npy")
centroids = np.load("face_centroids.npy")
thresholds = np.load("face_thresholds.npy")

embedding_model = tf.keras.models.Model(
    inputs=model.input,
    outputs=model.get_layer('embedding').output
)

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

EMB_BUFFER_SIZE = 5
SOFT_CONF = 0.55
SOFT_MARGIN = 0.12

cap = cv2.VideoCapture(0)
emb_buffer = deque(maxlen=EMB_BUFFER_SIZE)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # SAMAKAN PREPROCESSING DENGAN TRAINING
    enhanced = apply_clahe(frame)
    rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    display_name = "No Face"
    conf = 0.0
    dist = 0.0
    nearest_name = "-"

    if results.multi_face_landmarks:
        face_landmarks = results.multi_face_landmarks[0]

        lms = np.array(
            [[lm.x, lm.y, lm.z] for lm in face_landmarks.landmark],
            dtype=np.float32
        )

        aligned = align_landmarks(lms)
        norm = (aligned - mean) / std
        x_input = np.expand_dims(norm, axis=0)

        pred = model.predict(x_input, verbose=0)[0]
        emb = embedding_model.predict(x_input, verbose=0)[0]

        emb_buffer.append(emb)
        smoothed_emb = np.mean(emb_buffer, axis=0)

        # Cari kelas terdekat berdasarkan embedding distance
        dists = np.linalg.norm(centroids - smoothed_emb, axis=1)
        cls_idx = int(np.argmin(dists))
        dist = float(dists[cls_idx])
        nearest_name = le.classes_[cls_idx]

        sorted_pred = np.sort(pred)
        top1 = sorted_pred[-1]
        top2 = sorted_pred[-2]
        conf = float(pred[cls_idx])

        if dist <= thresholds[cls_idx]:
            display_name = le.classes_[cls_idx]
        elif top1 >= SOFT_CONF and (top1 - top2) >= SOFT_MARGIN:
            display_name = le.classes_[int(np.argmax(pred))]
        else:
            display_name = "Unknown"

        # Bounding box dari landmark
        h, w, _ = frame.shape
        xs = [lm.x * w for lm in face_landmarks.landmark]
        ys = [lm.y * h for lm in face_landmarks.landmark]
        x1 = int(min(xs))
        y1 = int(min(ys))
        x2 = int(max(xs))
        y2 = int(max(ys))

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        text = f"{display_name} ({conf*100:.1f}%, d={dist:.2f})"
        cv2.putText(
            frame, text, (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
        )

        print(
            f"[DEBUG] display={display_name} | nearest={nearest_name} | "
            f"dist={dist:.3f} | threshold={thresholds[cls_idx]:.3f} | "
            f"softmax={top1:.3f} (margin={top1-top2:.3f})"
        )
    else:
        emb_buffer.clear()

    cv2.imshow("MediaPipe + Embedding Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


---

### Catatan
- Kalau masih sering salah, coba turunkan/increase threshold percentile dari 95 menjadi 90 atau 99 tergantung seberapa ketat unknown detection yang diinginkan.
- Untuk performa lebih baik lagi, pertimbangkan pakai **pretrained face embedding** (FaceNet/ArcFace/DeepFace) + distance threshold, karena landmark-only CNN masih relatif lemah untuk perbedaan antar orang yang mirip.
